# 01. 市況データの取得

LSEG から週次レポート用の市況データを取得し、`data/processed/<output_ymd>/market/` に保存します。
パラメータを変えて、すべて実行してください。LSEG の認証情報 (`.env`) が必要です。

- 指数・株価・利回り・コモディティの時系列は `from_ymd` から `to_ymd` まで取得します。
- 決算日と経済指標は、取得した時点の予定を取ります (経済指標の値は `output_ymd` まで)。
- BM は `data/raw/bm/` のうち `output_ymd` 以前で最も新しい日のものを使います。

In [ ]:
output_ymd = 20261125  # 保存先の日付 (data/processed/<日付>/market/)。通常はレポートを作る日 (水曜)
from_ymd = 20260801  # 時系列の取得開始日。コモディティの図に 3 か月分使う
to_ymd = 20261124  # 時系列の取得終了日。通常は output_ymd の前日

In [ ]:
# em_market のコードを変更したとき、カーネルを再起動せずに反映する
%load_ext autoreload
%autoreload 2

import json

import pandas as pd

from em_market.market.client import LSEGClient
from em_market.paths import processed_dir
from em_market.reports.weekly import WeeklyReport, WeeklyReportConfig

output_date = pd.Timestamp(str(output_ymd))
start, end = pd.Timestamp(str(from_ymd)), pd.Timestamp(str(to_ymd))
assert start <= end, f"from_ymd が to_ymd より後です: {from_ymd} > {to_ymd}"

# 設定は configs/weekly_report.yaml
report = WeeklyReport(base_date=output_date, config=WeeklyReportConfig.load())
print(f"BM: data/raw/bm/{report.bm.attrs['date']}.csv ({len(report.bm)} 銘柄)")

## 取得

In [ ]:
with LSEGClient() as client:
    data = report.fetch(client, start=start, end=end)

## 保存

In [ ]:
output_dir = processed_dir(output_date, "market")
paths = data.save(output_dir)
meta = {"from_ymd": from_ymd, "to_ymd": to_ymd}
(output_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"Saved {len(paths) + 1} files to {output_dir}")

In [ ]:
# 保存したデータの概要
frames = {f"index_prices_{g}": df for g, df in data.index_prices.items()}
frames |= {
    name: getattr(data, name)
    for name in [
        "stock_prices",
        "us_yields",
        "short_yields",
        "long_yields",
        "commodity_prices",
        "earnings_dates",
        "econ_calendar",
    ]
}
pd.DataFrame(
    {name: {"rows": df.shape[0], "columns": df.shape[1]} for name, df in frames.items()}
).T